# Feed-Forward Network


In [1]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

LECTURE_NOTE_PATH = Path('/Users/montekkundan/Library/Mobile Documents/iCloud~md~obsidian/Documents/lectures/LLM/concepts/Feed-Forward Network.md')
print(LECTURE_NOTE_PATH)


/Users/montekkundan/Library/Mobile Documents/iCloud~md~obsidian/Documents/lectures/LLM/concepts/Feed-Forward Network.md


## Demo A: the same FFN is applied independently at every token


In [2]:
class FFN(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
    def forward(self, x):
        return self.net(x)

torch.manual_seed(0)
x = torch.randn(2, 3, 8)
ffn = FFN(8, 32)
y = ffn(x)
print('full batch shape:', y.shape)
print('tokenwise equality:', torch.allclose(y[0, 1], ffn(x[0, 1:2])[0]))


full batch shape: torch.Size([2, 3, 8])
tokenwise equality: True


## Demo B: parameter count grows with d_ff


In [3]:
for d_model in [256, 512]:
    for d_ff in [d_model * 2, d_model * 4]:
        params = 2 * d_model * d_ff
        print(f'd_model={d_model}, d_ff={d_ff} -> approx params={params:,}')


d_model=256, d_ff=512 -> approx params=262,144
d_model=256, d_ff=1024 -> approx params=524,288
d_model=512, d_ff=1024 -> approx params=1,048,576
d_model=512, d_ff=2048 -> approx params=2,097,152


## Demo C: activation choice changes the hidden map


In [4]:
x = torch.linspace(-3, 3, steps=9)
print('x    :', x.tolist())
print('relu :', F.relu(x).tolist())
print('gelu :', [round(v, 4) for v in F.gelu(x).tolist()])
print('silu :', [round(v, 4) for v in F.silu(x).tolist()])


x    : [-3.0, -2.25, -1.5, -0.75, 0.0, 0.75, 1.5, 2.25, 3.0]
relu : [0.0, 0.0, 0.0, 0.0, 0.0, 0.75, 1.5, 2.25, 3.0]
gelu : [-0.004, -0.0275, -0.1002, -0.17, 0.0, 0.58, 1.3998, 2.2225, 2.996]
silu : [-0.1423, -0.2145, -0.2736, -0.2406, 0.0, 0.5094, 1.2264, 2.0355, 2.8577]


## Demo D: gated FFNs add a modulation branch


In [5]:
class SwiGLU(nn.Module):
    def __init__(self, d_model: int, hidden: int):
        super().__init__()
        self.gate = nn.Linear(d_model, hidden)
        self.up = nn.Linear(d_model, hidden)
        self.down = nn.Linear(hidden, d_model)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

standard_params = sum(p.numel() for p in FFN(512, 2048).parameters())
swiglu_params = sum(p.numel() for p in SwiGLU(512, int(2048 * 2 / 3)).parameters())
print('standard params:', standard_params)
print('swiglu params  :', swiglu_params)


standard params: 2099712
swiglu params  : 2099882


## Demo E: attention mixes tokens, FFN mixes channels


In [6]:
attn_weights = torch.tensor([[0.7, 0.3], [0.2, 0.8]])
values = torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
after_attention = attn_weights @ values
after_ffn = FFN(3, 12)(after_attention)
print('after attention:')
print(after_attention)
print('after FFN:')
print(after_ffn)


after attention:
tensor([[0.7000, 0.3000, 0.0000],
        [0.2000, 0.8000, 0.0000]])
after FFN:
tensor([[-0.0576, -0.0999,  0.0431],
        [-0.0809, -0.0951,  0.1365]], grad_fn=<AddmmBackward0>)


## Rasbt and nanochat


In [7]:
refs = {
    'rasbt': [
        'https://github.com/rasbt/LLMs-from-scratch/blob/main/ch04/01_main-chapter-code/gpt.py',
    ],
    'nanochat': [
        'https://github.com/karpathy/nanochat/blob/master/nanochat/gpt.py',
    ],
}
for name, files in refs.items():
    print(name.upper())
    for f in files:
        print(' -', f)
    print()


RASBT
 - https://github.com/rasbt/LLMs-from-scratch/blob/main/ch04/01_main-chapter-code/gpt.py

NANOCHAT
 - https://github.com/karpathy/nanochat/blob/master/nanochat/gpt.py

